# Project Title: Global Comparison — Intelligent Multi-Doc Audit & Analysis System

This project is an advanced document processing engine powered by Ollama (Qwen3-Coder:30B), specifically engineered to tackle the challenges of deep analysis for long-form texts and horizontal comparisons across multiple files. It integrates sliding window chunking, multi-level cascading summarization, and cross-document stance auditing to automatically extract key insights, sentiments, and conflicting viewpoints from unstructured data.

In [ ]:
# Check AMD GPU availability
import torch
print(f'ROCm available: {torch.cuda.is_available()}')
print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None"}')

In [ ]:
# author:GuanHua Yu
import httpx
import json
import asyncio
import ipywidgets as widgets

OLLAMA_URL = "http://open-webui-ollama.open-webui:11434/api/generate"
MODEL_NAME = "qwen3-coder:30b"
# 1 Chinese character is usually > 1 token; do not set to max model context
CHUNK_SIZE = 15000 
OVERLAP = 500      

def split_text(text):
    """ Sliding window slicing logic """
    chunks, start = [], 0
    if not text: return [""]
    while start < len(text):
        chunks.append(text[start : start + CHUNK_SIZE])
        start += (CHUNK_SIZE - OVERLAP)
        if start >= len(text): break
    return chunks

async def analyze_text_task_stream(content, target_output, task_type="ANALYSIS"):
    """
    Core Engine: Supports ANALYSIS, SUMMARY, and COMPARISON.
    Contains prompts, parameter settings, and asynchronous httpx calls.
    """
    prompts = {
        "ANALYSIS": "You are a strictly controlled deep analysis engine. Thought steps: 1. Identify core events 2. Analyze tone/emotion 3. Extract keywords.",
        "SUMMARY": "You are a report aggregation expert. Thought steps: 1. Compare similarities/differences between sections 2. De-duplicate while keeping key details 3. Synthesize a coherent summary.",
        "COMPARISON": "You are a Chief Audit Analysis Officer. Thought steps: 1. Identify positional conflicts between files 2. Summarize shared information 3. Evaluate the unique value of each file."
    }
    
    few_shot = (
        "\n--- Example ---\n"
        "Input: Recently tried the ROCm solution; installation had some issues but speed increased by 40%.\n"
        "Output:\n<Thought>Mentioned technical solution and performance; contains negative details but overall evaluation is positive.</Thought>\n"
        "## 📝 Core Summary\nROCm solution has installation hurdles but significant performance gains; evaluation is positive.\n"
        "## 🎭 Sentiment\nPositive\n"
        "## 🔑 Core Keywords\nROCm, Acceleration, Performance\n"
        "--- End of Example ---\n\n"
    )

    instruction = (
        f"{prompts.get(task_type, prompts['ANALYSIS'])}\n"
        # Only inject examples during basic analysis
        f"{few_shot if task_type == 'ANALYSIS' else ''}"
        "**Mandatory Output Format:**\n"
        "<Thought>\n[Record your comparison/reasoning logic here]\n</Thought>\n\n"
    )
    
    if task_type == "COMPARISON":
        instruction += "## 📊 Global Comparison Overview\n[Summarize overall situation]\n## 🤝 Common Core Points\n[List consistent information]\n## 🔄 Key Difference Analysis\n[Analyze conflicts in stance or data]"
    else:
        instruction += "## 📝 Core Summary\n[Within 100 words]\n## 🎭 Sentiment\n[Positive/Neutral/Negative]\n## 🔑 Core Keywords\n[3 words, comma-separated]"

    payload = {
        "model": MODEL_NAME,
        "prompt": f"{instruction}\n\nContent to process:\n{content}",
        "stream": True,
        "options": {"num_ctx": 32000, "temperature": 0.0}
    }

    full_response, buffer = "", bytearray()

    # Increased timeout to handle complex comparison tasks
    async with httpx.AsyncClient(timeout=300) as client:
        try:
            async with client.stream("POST", OLLAMA_URL, json=payload) as response:
                if response.status_code != 200:
                    target_output.append_stdout(f"\n❌ API Error: {response.status_code}")
                    return False, f"Error {response.status_code}"
                
                # Asynchronously iterate through bytes; chunk sizes are not fixed
                async for chunk in response.aiter_bytes():
                    buffer.extend(chunk)
                    # A single chunk may contain multiple lines separated by newlines
                    while b'\n' in buffer:
                        line, rest = buffer.split(b'\n', 1)
                        buffer = bytearray(rest)
                        try:
                            line_json = json.loads(line.decode('utf-8'))
                            part = line_json.get("response", "")
                            target_output.append_stdout(part)
                            full_response += part
                        except: continue
            return (True, full_response) if full_response else (False, "Content is empty")
        except Exception as e:
            target_output.append_stdout(f"\n❌ Communication Exception: {str(e)}")
            return False, str(e)

async def process_single_file(filename, content, target_output):
    """ Single-file cascading processing pipeline """
    target_output.append_stdout(f"\n\n📄 Processing: {filename} ({len(content)} characters)...\n")
    chunks = split_text(content)
    chunk_reports = []
    
    for i, chunk in enumerate(chunks):
        if len(chunks) > 1: 
            target_output.append_stdout(f"\n[Segment {i+1}/{len(chunks)}] ")
        success, res = await analyze_text_task_stream(chunk, target_output, task_type="ANALYSIS")
        if not success: return None
        chunk_reports.append(res)
    
    # Cascading aggregation: if multiple chunks exist, summarize them
    if len(chunk_reports) > 1:
        target_output.append_stdout(f"\n\n🔄 Aggregating multiple segments for 【{filename}】...\n")
        success, final_sum = await analyze_text_task_stream("\n".join(chunk_reports), target_output, task_type="SUMMARY")
        return final_sum if success else None
    else:
        return chunk_reports[0]

async def run_global_comparison(b):
    """ Button logic covering single-file and multi-file processing """
    output.clear_output()
    files_count = len(uploader.value)

    if files_count < 1:
        output.append_stdout("⚠️ Please upload files first.")
        return
    
    output.append_stdout(f"🚀 Launching Pipeline: {files_count} files total\n")
    all_file_reports = []
    
    # Internal slicing and summarization for each file
    for file_info in uploader.value:
        fname = file_info['name']
        raw_data = file_info['content']

        # Use tobytes method for decoding
        try:
            text = raw_data.tobytes().decode('utf-8')
        except:
            output.append_stdout(f"❌ Failed to decode file {fname}\n")
            continue
            
        report = await process_single_file(fname, text, output)
        if report:
            all_file_reports.append(f"### File Source: {fname}\n{report}")
            output.append_stdout(f"\n\n✅ Local processing for {fname} complete.\n" + "-"*30)

    # If multiple files exist, perform global cross-comparison analysis
    if len(all_file_reports) >= 2:
        output.append_stdout("\n" + "█"*15 + " Generating Global Comparison Report " + "█"*15 + "\n\n")
        comparison_input = "\n\n".join(all_file_reports)
        success, detail = await analyze_text_task_stream(comparison_input, output, task_type="COMPARISON")
        
        if success:
            output.append_stdout(f"\n\n{'='*40}\n✨ Cross-document horizontal comparison complete!")
        else:
            output.append_stdout(f"\n\n❌ Comparison failed: {detail}")
    else:
        output.append_stdout("\n💡 Insufficient file count; skipping horizontal comparison analysis.")

# User Interface UI Logic
uploader = widgets.FileUpload(accept='.txt,.md', multiple=True, description="Select Files")
start_btn = widgets.Button(description="Start Pipeline", button_style='primary', layout=widgets.Layout(width='200px'))
# Use asyncio to ensure UI thread is not blocked
start_btn.on_click(lambda b: asyncio.create_task(run_global_comparison(b)))
output = widgets.Output()

display(widgets.VBox([
    widgets.HTML("<h3>Cross-Document Intelligent Audit & Comparison System (Global Comparison)</h3>"),
    uploader, 
    start_btn, 
    output
]))